# MNIST Dimensionality Reduction and Digit Recognition

This notebook covers:

- **Part A:** Data Understanding and Preprocessing
- **Part B:** Principal Component Analysis (PCA)
- **Part C:** t-SNE Visualization
- **Part D:** Digit Recognition Analysis using KNN

The notebook uses the provided `mnist_train.csv` and `mnist_test.csv` files. It does **not** require downloading MNIST through TensorFlow.

> **Note:** t-SNE and KNN can be computationally expensive on the complete MNIST dataset. The notebook therefore uses configurable sample sizes for visualization/t-SNE while keeping the classifier comparison reproducible.


In [ ]:
# Imports and configuration
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
np.random.seed(42)

TRAIN_FILE = "mnist_train.csv"
TEST_FILE = "mnist_test.csv"

# If the notebook is opened in the same environment as the uploaded files,
# these paths work automatically.
if not os.path.exists(TRAIN_FILE) and os.path.exists("/mnt/data/mnist_train.csv"):
    TRAIN_FILE = "/mnt/data/mnist_train.csv"

if not os.path.exists(TEST_FILE) and os.path.exists("/mnt/data/mnist_test.csv"):
    TEST_FILE = "/mnt/data/mnist_test.csv"

# Configurable settings
TSNE_SAMPLES = 5000
TSNE_PERPLEXITIES = [5, 30, 50]
KNN_NEIGHBORS = 5

print("Train file:", TRAIN_FILE)
print("Test file :", TEST_FILE)


## Part A — Data Understanding and Preprocessing

In [ ]:
# Load the CSV datasets
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

print("Training shape:", train_df.shape)
print("Testing shape :", test_df.shape)

display(train_df.head())


In [ ]:
# Separate labels and image pixels
def get_label_column(df):
    for col in ["label", "Label", "target", "Target", "class", "Class"]:
        if col in df.columns:
            return col
    # MNIST CSV files normally store the label in the first column.
    return df.columns[0]

label_col_train = get_label_column(train_df)
label_col_test = get_label_column(test_df)

y_train = train_df[label_col_train].to_numpy()
y_test = test_df[label_col_test].to_numpy()

X_train = train_df.drop(columns=[label_col_train]).to_numpy(dtype=np.float32)
X_test = test_df.drop(columns=[label_col_test]).to_numpy(dtype=np.float32)

print("Label column:", label_col_train)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


In [ ]:
# Dataset information
unique_labels, label_counts = np.unique(y_train, return_counts=True)

print(f"Number of training samples: {X_train.shape[0]}")
print(f"Number of test samples: {X_test.shape[0]}")
print(f"Number of classes: {len(unique_labels)}")
print(f"Image dimensions: {X_train.shape[1]} pixels = 28 x 28")

print("\nUnique labels:", unique_labels)
print("Number of samples per label:")
for label, count in zip(unique_labels, label_counts):
    print(f"{label}: {count}")


In [ ]:
# Normalize pixel values to [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

print("Minimum pixel value:", X_train.min())
print("Maximum pixel value:", X_train.max())


In [ ]:
# Visualize at least 10 handwritten digit images
plt.figure(figsize=(12, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i].reshape(28, 28), cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")

plt.suptitle("Sample MNIST Handwritten Digits")
plt.tight_layout()
plt.show()


## Part B — Principal Component Analysis (PCA)

In [ ]:
# PCA with all components for explained variance analysis
pca_all = PCA()
pca_all.fit(X_train)

explained_variance = pca_all.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("Number of principal components:", len(explained_variance))
print("Variance explained by first 10 components:")
for i, value in enumerate(explained_variance[:10], start=1):
    print(f"PC{i}: {value:.6f} ({value*100:.2f}%)")


In [ ]:
# Components required for 90%, 95% and 99% variance
variance_targets = [0.90, 0.95, 0.99]

for target in variance_targets:
    n_components = np.argmax(cumulative_variance >= target) + 1
    print(f"Components required for {target*100:.0f}% variance: {n_components}")


In [ ]:
# Scree Plot
plt.figure(figsize=(10, 5))
plt.plot(
    range(1, len(explained_variance) + 1),
    explained_variance,
    linewidth=1
)
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Cumulative Explained Variance Plot
plt.figure(figsize=(10, 5))
plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    linewidth=2
)

for target in variance_targets:
    n = np.argmax(cumulative_variance >= target) + 1
    plt.axhline(target, linestyle="--", label=f"{target*100:.0f}% -> {n} components")
    plt.axvline(n, linestyle="--")

plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# PCA reductions to 2 and 50 dimensions
pca_2 = PCA(n_components=2, random_state=42)
pca_50 = PCA(n_components=50, random_state=42)

start = time.perf_counter()
X_train_pca_2 = pca_2.fit_transform(X_train)
X_test_pca_2 = pca_2.transform(X_test)
pca2_time = time.perf_counter() - start

start = time.perf_counter()
X_train_pca_50 = pca_50.fit_transform(X_train)
X_test_pca_50 = pca_50.transform(X_test)
pca50_time = time.perf_counter() - start

print("2D PCA shape :", X_train_pca_2.shape)
print("50D PCA shape:", X_train_pca_50.shape)
print(f"2D PCA time  : {pca2_time:.3f} seconds")
print(f"50D PCA time : {pca50_time:.3f} seconds")
print(f"50D cumulative variance: {pca_50.explained_variance_ratio_.sum()*100:.2f}%")


In [ ]:
# 2D PCA Projection
plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    X_train_pca_2[:, 0],
    X_train_pca_2[:, 1],
    c=y_train,
    cmap="tab10",
    s=5,
    alpha=0.5
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("2D PCA Projection of MNIST")
plt.colorbar(scatter, label="Digit")
plt.show()


In [ ]:
# Optional reconstruction using 50 principal components
X_train_pca_recon = pca_50.inverse_transform(X_train_pca_50)

plt.figure(figsize=(12, 5))

for i in range(5):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i].reshape(28, 28), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    plt.subplot(2, 5, i + 6)
    plt.imshow(X_train_pca_recon[i].reshape(28, 28), cmap="gray")
    plt.title("PCA 50 Reconstruction")
    plt.axis("off")

plt.tight_layout()
plt.show()


## Part C — t-SNE Visualization

t-SNE is computationally expensive on all 60,000 training images. For practical execution, a reproducible subset is used.

Two experiments are performed:

1. t-SNE directly on the original 784-dimensional data.
2. t-SNE after reducing the data to 50 dimensions using PCA.

The same sample indices are used for both experiments so the comparison is fair.


In [ ]:
# Select a reproducible subset for t-SNE
tsne_n = min(TSNE_SAMPLES, len(X_train))
rng = np.random.default_rng(42)
tsne_indices = rng.choice(len(X_train), size=tsne_n, replace=False)

X_tsne_original = X_train[tsne_indices]
X_tsne_pca50 = X_train_pca_50[tsne_indices]
y_tsne = y_train[tsne_indices]

print("t-SNE samples:", tsne_n)
print("Original t-SNE input:", X_tsne_original.shape)
print("PCA-50 t-SNE input   :", X_tsne_pca50.shape)


In [ ]:
# t-SNE helper function
def run_tsne(X, perplexity, title):
    start = time.perf_counter()

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        max_iter=1000,
        random_state=42
    )

    X_embedded = tsne.fit_transform(X)
    elapsed = time.perf_counter() - start

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        X_embedded[:, 0],
        X_embedded[:, 1],
        c=y_tsne,
        cmap="tab10",
        s=7,
        alpha=0.65
    )
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.title(f"{title} | Perplexity = {perplexity}")
    plt.colorbar(scatter, label="Digit")
    plt.show()

    print(f"{title}, perplexity={perplexity}: {elapsed:.2f} seconds")
    return X_embedded, elapsed


In [ ]:
# t-SNE on original 784-dimensional data
tsne_original_results = {}

for perplexity in TSNE_PERPLEXITIES:
    embedding, elapsed = run_tsne(
        X_tsne_original,
        perplexity,
        "t-SNE on Original 784D Data"
    )
    tsne_original_results[perplexity] = (embedding, elapsed)


In [ ]:
# t-SNE on PCA-reduced 50-dimensional data
tsne_pca_results = {}

for perplexity in TSNE_PERPLEXITIES:
    embedding, elapsed = run_tsne(
        X_tsne_pca50,
        perplexity,
        "t-SNE on PCA 50D Data"
    )
    tsne_pca_results[perplexity] = (embedding, elapsed)


### t-SNE Perplexity Comparison

Typical interpretation:

- **Low perplexity** (e.g. 5): emphasizes very local neighborhoods and can create many small clusters.
- **Medium perplexity** (e.g. 30): often provides a balanced view of local and global structure.
- **Higher perplexity** (e.g. 50): considers broader neighborhoods and may produce smoother, larger clusters.

The best perplexity should be selected based on the clarity and stability of the digit clusters rather than accuracy alone.


## Part D — Digit Recognition Analysis

In [ ]:
# KNN on original 784-dimensional data
print("Training and evaluating KNN with ORIGINAL data...")

knn_original = KNeighborsClassifier(n_neighbors=KNN_NEIGHBORS)

start_train = time.perf_counter()
knn_original.fit(X_train, y_train)
original_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()
y_pred_original = knn_original.predict(X_test)
original_prediction_time = time.perf_counter() - start_pred

accuracy_original = accuracy_score(y_test, y_pred_original)

print(f"Accuracy: {accuracy_original:.4f}")
print(f"Training time: {original_train_time:.3f} seconds")
print(f"Prediction time: {original_prediction_time:.3f} seconds")


In [ ]:
# KNN on PCA 50-dimensional data
print("Training and evaluating KNN with PCA-50 data...")

knn_pca = KNeighborsClassifier(n_neighbors=KNN_NEIGHBORS)

start_train = time.perf_counter()
knn_pca.fit(X_train_pca_50, y_train)
pca_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()
y_pred_pca = knn_pca.predict(X_test_pca_50)
pca_prediction_time = time.perf_counter() - start_pred

accuracy_pca = accuracy_score(y_test, y_pred_pca)

print(f"Accuracy: {accuracy_pca:.4f}")
print(f"Training time: {pca_train_time:.3f} seconds")
print(f"Prediction time: {pca_prediction_time:.3f} seconds")


In [ ]:
# Performance comparison
results = pd.DataFrame({
    "Dataset": ["Original 784D", "PCA 50D"],
    "Dimensions": [X_train.shape[1], X_train_pca_50.shape[1]],
    "Accuracy": [accuracy_original, accuracy_pca],
    "Training Time (s)": [original_train_time, pca_train_time],
    "Prediction Time (s)": [original_prediction_time, pca_prediction_time],
    "Memory (MB)": [
        X_train.nbytes / (1024**2),
        X_train_pca_50.nbytes / (1024**2)
    ]
})

display(results)


In [ ]:
# Accuracy and timing comparison plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(results["Dataset"], results["Accuracy"])
axes[0].set_title("Classification Accuracy")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)

axes[1].bar(results["Dataset"], results["Training Time (s)"])
axes[1].set_title("Training Time")
axes[1].set_ylabel("Seconds")

axes[2].bar(results["Dataset"], results["Prediction Time (s)"])
axes[2].set_title("Prediction Time")
axes[2].set_ylabel("Seconds")

plt.tight_layout()
plt.show()


In [ ]:
# Memory comparison
print("Memory Usage:")
print(f"Original data: {X_train.nbytes / (1024**2):.2f} MB")
print(f"PCA 50D data : {X_train_pca_50.nbytes / (1024**2):.2f} MB")

memory_reduction = (
    1 - X_train_pca_50.nbytes / X_train.nbytes
) * 100

print(f"Memory reduction: {memory_reduction:.2f}%")


In [ ]:
# Final analysis summary
accuracy_change = accuracy_pca - accuracy_original
prediction_speedup = original_prediction_time / pca_prediction_time if pca_prediction_time > 0 else np.nan

print("FINAL ANALYSIS")
print("=" * 60)
print(f"Original dimensions : {X_train.shape[1]}")
print(f"PCA dimensions      : {X_train_pca_50.shape[1]}")
print(f"Original accuracy   : {accuracy_original:.4f}")
print(f"PCA accuracy        : {accuracy_pca:.4f}")
print(f"Accuracy change     : {accuracy_change:+.4f}")
print(f"Original prediction : {original_prediction_time:.3f} s")
print(f"PCA prediction      : {pca_prediction_time:.3f} s")
print(f"Prediction speedup  : {prediction_speedup:.2f}x")
print(f"Memory reduction    : {memory_reduction:.2f}%")

print("\nConclusion:")
if accuracy_pca >= accuracy_original - 0.02:
    print("PCA retained most of the classification performance while substantially reducing dimensionality.")
else:
    print("PCA caused a noticeable reduction in classification accuracy, indicating that some discriminative information was lost.")

print("Dimensionality reduction can reduce memory usage and distance-computation cost,")
print("while the exact accuracy/time trade-off depends on the number of retained components.")


## Overall Conclusion

PCA transforms the original 784 pixel features into a smaller set of orthogonal components that preserve most of the data variance. The cumulative explained variance determines how many components are needed for 90%, 95%, and 99% variance retention.

The 2D PCA projection provides a quick visual overview of digit separation, while t-SNE generally gives a more detailed nonlinear visualization of local clusters. Changing perplexity changes the balance between local and broader neighborhood structure.

For classification, PCA can substantially reduce the dimensionality and memory requirement of MNIST. The final accuracy, training time, and prediction time measurements above provide the experimental evidence for evaluating that trade-off.
